In [375]:
import pandas as pd
import os

In [376]:
provincias = {
    "Madrid":"../data/raw/Madrid",
    "Barcelona":"../data/raw/Barcelona",
    "Valencia":"../data/raw/Valencia"
}
dfs_por_provincia = {
    "Madrid": [],
    "Barcelona": [],
    "Valencia": []
}

for provincia, carpeta in provincias.items():
    for archivo in os.listdir(carpeta):
        if archivo.endswith(".csv"):
            ruta = os.path.join(carpeta, archivo)
            df = pd.read_csv(ruta, encoding="latin1", sep=";", decimal=",",thousands=".")
            df["Provincia"] = provincia
            dfs_por_provincia[provincia].append(df)


In [377]:
df_madrid = pd.concat(dfs_por_provincia["Madrid"])
df_barcelona = pd.concat(dfs_por_provincia["Barcelona"])
df_valencia = pd.concat(dfs_por_provincia["Valencia"])


In [378]:
df_barcelona = df_barcelona.copy()
df_barcelona["Geografía"].head(40)

#Valores unicos ordenados
sorted(df_barcelona["Geografía"].unique())
#Numero de filas con dos o mas espacios seguidos
df_barcelona["Geografía"].str.contains(r"\s{2,}").sum()
#Numero de filas que comienzan con un espacio
df_barcelona["Geografía"].str.startswith(" ").sum()
#Frecuencias de los valores
df_barcelona["Geografía"].str.lower().value_counts()
#Normaliza espacios y luego cuenta(ver si hay variantes del mismo muncipio)
df_barcelona["Geografía"].str.strip().str.lower().value_counts()
#Estadisticas de longitud de los str
df_barcelona["Geografía"].str.len().describe().T
#Cuenta valores nulos
df_barcelona["Geografía"].isna().sum()
#Cuenta strings vacios que no son nulos
df_barcelona["Geografía"].eq("").sum()
#str que empiezan con:
df_cp = df_barcelona[~df_barcelona["Geografía"].str.startswith("- Municipio de")]
#Detectamos simbolos extraños       
df_barcelona["Geografía"].str.contains(r"[-()_/]").sum() #->472
#Detectamos los valores con estos simbolos
df_barcelona[df_barcelona["Geografía"].str.contains(r"[-()_/]")]["Geografía"].unique()



array(['- Municipio de Badalona', '- Municipio de Barberà del Vallès',
       '- Municipio de Barcelona', '- Municipio de Castelldefels',
       '- Municipio de Cerdanyola del Vallès',
       '- Municipio de Cornellà de Llobregat',
       '- Municipio de Esplugues de Llobregat', '- Municipio de Gavà',
       '- Municipio de Granollers',
       "- Municipio de Hospitalet de Llobregat (L')",
       '- Municipio de Igualada', '- Municipio de Manresa',
       '- Municipio de Mataró', '- Municipio de Mollet del Vallès',
       '- Municipio de Montcada i Reixac',
       '- Municipio de Prat de Llobregat (El)', '- Municipio de Ripollet',
       '- Municipio de Rubí', '- Municipio de Sabadell',
       '- Municipio de Sant Adrià de Besòs',
       '- Municipio de Sant Boi de Llobregat',
       '- Municipio de Sant Cugat del Vallès',
       '- Municipio de Sant Feliu de Llobregat',
       '- Municipio de Sant Joan Despí',
       '- Municipio de Sant Pere de Ribes',
       '- Municipio de Santa Co

In [379]:
df_barcelona["Periodos:"].value_counts()

Periodos:
enero-septiembre 2025    931
enero-junio 2025         931
enero-marzo 2025         931
enero-septiembre 2024    912
enero-marzo 2024         912
enero-diciembre 2023     912
enero-junio 2024         912
enero-diciembre 2024     912
Enero-junio 2023         893
Enero-marzo 2023         893
enero-septiembre 2023    893
Enero-diciembre 2022     893
Enero-marzo 2021         705
Enero-marzo 2022         705
Enero-septiembre 2022    705
Enero-junio 2022         705
Enero-diciembre 2021     705
Enero-junio 2021         705
Enero-septiembre 2021    705
Enero-diciembre 2018     465
Enero-septiembre 2020    465
Enero-junio 2019         465
Enero-marzo 2020         465
Enero-marzo 2019         465
Enero-diciembre 2019     465
Enero-septiembre 2019    465
Enero-diciembre 2020     465
Enero-junio 2020         465
Enero-diciembre 2017     434
Enero-junio 2018         434
Enero-marzo 2018         434
Enero-septiembre 2018    434
Enero-marzo 2017         420
Enero-junio 2017         420
Ener

In [380]:
df_barcelona.duplicated().sum()
df_valencia.duplicated().sum()
df_madrid.duplicated().sum()

np.int64(0)

In [381]:
def limpiar_geografia_general(df):

    df = df.copy()

    #Eliminar códigos postales (primeros 5 caracteres numéricos)
    df["Geografía"] = df["Geografía"].apply(lambda x: x[5:] if isinstance(x,str) and x[:5].isdigit() else x)
    
    #Eliminar el prefijo "- Municipio de"
    prefijos = ["- municipio de", 
                "-municipio de", 
                "- municipo de", 
                "-municipo de", 
                "municipio de", 
                "municipo de"]
    
    for p in prefijos:
        df["Geografía"] = df["Geografía"].str.replace(p,"", case=False, regex=False)
        
    #Eliminar espacios al inicio y final
    df["Geografía"] = df["Geografía"].str.strip()
    
    #Corrección puntual detectada en Madrid
    df["Geografía"] = df["Geografía"].replace("Rozas de Madrid (Las)","Las Rozas de Madrid")

    #Nos deshacemos de 2 provincias que no tienen que estar en Madrid
    df = df[~df["Geografía"].isin(["Provincia de ALMERÍA", "ANDALUCÍA"])]

    return df

df_limpieza_geografia = limpiar_geografia_general(df_madrid)
df_limpieza_geografia




,Geografía,Tipología penal,Periodos:,Total,Provincia
0,Alcalá de Henares,1.-Homicidios dolosos y asesinatos consumados,Enero-junio 2020,0,Madrid
1,Alcalá de Henares,2.-Homicidios dolosos y asesinatos en grado te...,Enero-junio 2020,1,Madrid
2,Alcalá de Henares,3.-Delitos graves y menos graves de lesiones y...,Enero-junio 2020,17,Madrid
3,Alcalá de Henares,4.-Secuestro,Enero-junio 2020,1,Madrid
4,Alcalá de Henares,5.-Delitos contra la libertad e indemnidad sexual,Enero-junio 2020,18,Madrid
...,...,...,...,...,...
155,Valdemoro,4.-ROBOS CON FUERZA EN DOMICILIOS (EU),Enero-marzo 2016,36,Madrid
156,Valdemoro,5.-SUSTRACCIÓN VEHÍCULOS A MOTOR (EU),Enero-marzo 2016,20,Madrid
157,Valdemoro,6.-TRÁFICO DE DROGAS (EU),Enero-marzo 2016,3,Madrid
158,Valdemoro,7.-DAÑOS,Enero-marzo 2016,119,Madrid


In [382]:
pd.set_option("display.max_rows",200)

df_prueba = pd.DataFrame(df_limpieza_geografia["Geografía"].unique())
df_prueba

,0
0,Alcalá de Henares
1,Alcobendas
2,Alcorcón
3,Aranjuez
4,Arganda del Rey
5,Arroyomolinos
6,Boadilla del Monte
7,Colmenar Viejo
8,Collado Villalba
9,Coslada


In [383]:
def limpiar_tipologia_penal(df):
    #Eliminamos tipología penal basura
    patron_basura = df["Tipología penal"].str.contains(r"TOTAL|Resto|I\.|II\.|III\.|EU|ciber|informátic",case=False, na=False)

    return df[~patron_basura].copy()

df_tipologia_penal_limpia = limpiar_tipologia_penal(df_limpieza_geografia)
df_tipologia_penal_limpia


,Geografía,Tipología penal,Periodos:,Total,Provincia
0,Alcalá de Henares,1.-Homicidios dolosos y asesinatos consumados,Enero-junio 2020,0,Madrid
1,Alcalá de Henares,2.-Homicidios dolosos y asesinatos en grado te...,Enero-junio 2020,1,Madrid
2,Alcalá de Henares,3.-Delitos graves y menos graves de lesiones y...,Enero-junio 2020,17,Madrid
3,Alcalá de Henares,4.-Secuestro,Enero-junio 2020,1,Madrid
4,Alcalá de Henares,5.-Delitos contra la libertad e indemnidad sexual,Enero-junio 2020,18,Madrid
...,...,...,...,...,...
143,San Sebastián de los Reyes,8.-HURTOS,Enero-marzo 2016,584,Madrid
150,Torrejón de Ardoz,7.-DAÑOS,Enero-marzo 2016,217,Madrid
151,Torrejón de Ardoz,8.-HURTOS,Enero-marzo 2016,416,Madrid
158,Valdemoro,7.-DAÑOS,Enero-marzo 2016,119,Madrid


In [384]:
mapa_normalizacion_valencia = {
    "Alboraia/Alboraya": "Alboraia / Alboraya",
    "Alboraya": "Alboraia / Alboraya",
    "Almazora/Almassora": "Almazora/Almassora",
    "Almassora": "Almazora/Almassora",
    "Sagunto/Sagunt": "Sagunto / Sagunt",
    "Valencia": "Valencia",
    "València": "Valencia"}


def normalizacion_municipios(df, mapa_municipios):
    
    df["Geografía"] = df["Geografía"].replace(mapa_municipios)

    return df

df_municipios_normalizados = normalizacion_municipios(df_tipologia_penal_limpia,mapa_normalizacion_valencia)
df_municipios_normalizados



,Geografía,Tipología penal,Periodos:,Total,Provincia
0,Alcalá de Henares,1.-Homicidios dolosos y asesinatos consumados,Enero-junio 2020,0,Madrid
1,Alcalá de Henares,2.-Homicidios dolosos y asesinatos en grado te...,Enero-junio 2020,1,Madrid
2,Alcalá de Henares,3.-Delitos graves y menos graves de lesiones y...,Enero-junio 2020,17,Madrid
3,Alcalá de Henares,4.-Secuestro,Enero-junio 2020,1,Madrid
4,Alcalá de Henares,5.-Delitos contra la libertad e indemnidad sexual,Enero-junio 2020,18,Madrid
...,...,...,...,...,...
143,San Sebastián de los Reyes,8.-HURTOS,Enero-marzo 2016,584,Madrid
150,Torrejón de Ardoz,7.-DAÑOS,Enero-marzo 2016,217,Madrid
151,Torrejón de Ardoz,8.-HURTOS,Enero-marzo 2016,416,Madrid
158,Valdemoro,7.-DAÑOS,Enero-marzo 2016,119,Madrid


In [385]:
lista_municipios_valencia = [
    "Alaquàs",
    "Alboraia / Alboraya",
    "Aldaia",
    "Alfafar",
    "Algemesí",
    "Alzira",
    "Bétera",
    "Burjassot",
    "Carcaixent",
    "Catarroja",
    "Cullera",
    "Gandia",
    "Llíria",
    "Manises",
    "Mislata",
    "Moncada",
    "Oliva",
    "Ontinyent",
    "Paiporta",
    "Paterna",
    "Picassent",
    "Pobla de Vallbona (la)",
    "Puçol",
    "Quart de Poblet",
    "Requena",
    "Riba-roja de Túria",
    "Sagunto / Sagunt",
    "Silla",
    "Sueca",
    "Torrent",
    "Valencia",
    "Xàtiva",
    "Xirivella"]


def filtrar_municipios(df,lista_municipios):
    
    df = df[df["Geografía"].isin(lista_municipios)].copy()

    return df

df_municipios_filtrados = filtrar_municipios(df_municipios_normalizados, lista_municipios_valencia)
df_municipios_filtrados

,Geografía,Tipología penal,Periodos:,Total,Provincia


In [386]:
df_municipios_normalizados["Tipología penal"].value_counts()


Tipología penal
5.1.-Agresión sexual con penetración                                          1100
7.1.-Robos con fuerza en domicilios                                           1100
3.-Delitos graves y menos graves de lesiones y riña tumultuaria                669
1.-Homicidios dolosos y asesinatos consumados                                  669
4.-Secuestro                                                                   669
5.-Delitos contra la libertad e indemnidad sexual                              669
6.-Robos con violencia e intimidación                                          669
2.-Homicidios dolosos y asesinatos en grado tentativa                          669
7.- Robos con fuerza en domicilios, establecimientos y otras instalaciones     669
8.-Hurtos                                                                      669
9.-Sustracciones de vehículos                                                  669
10.-Tráfico de drogas                                                  

In [387]:
def normalizar_texto_tipologia_penal(df):
    df = df.copy()
    df["Tipología penal"] = (df["Tipología penal"]
                             .str.strip()
                             .str.lower()
                             .str.replace(r"\s+", " ", regex=True) #normaliza espacios
                             .str.strip()
                             )
    df["Tipología penal"] = df["Tipología penal"].str.title()
    return df

df_tipologia_penal_normalizado = normalizar_texto_tipologia_penal(df_municipios_normalizados)
df_tipologia_penal_normalizado

,Geografía,Tipología penal,Periodos:,Total,Provincia
0,Alcalá de Henares,1.-Homicidios Dolosos Y Asesinatos Consumados,Enero-junio 2020,0,Madrid
1,Alcalá de Henares,2.-Homicidios Dolosos Y Asesinatos En Grado Te...,Enero-junio 2020,1,Madrid
2,Alcalá de Henares,3.-Delitos Graves Y Menos Graves De Lesiones Y...,Enero-junio 2020,17,Madrid
3,Alcalá de Henares,4.-Secuestro,Enero-junio 2020,1,Madrid
4,Alcalá de Henares,5.-Delitos Contra La Libertad E Indemnidad Sexual,Enero-junio 2020,18,Madrid
...,...,...,...,...,...
143,San Sebastián de los Reyes,8.-Hurtos,Enero-marzo 2016,584,Madrid
150,Torrejón de Ardoz,7.-Daños,Enero-marzo 2016,217,Madrid
151,Torrejón de Ardoz,8.-Hurtos,Enero-marzo 2016,416,Madrid
158,Valdemoro,7.-Daños,Enero-marzo 2016,119,Madrid


In [388]:
df_tipologia_penal_normalizado["Tipología penal"].unique()


array(['1.-Homicidios Dolosos Y Asesinatos Consumados',
       '2.-Homicidios Dolosos Y Asesinatos En Grado Tentativa',
       '3.-Delitos Graves Y Menos Graves De Lesiones Y Riña Tumultuaria',
       '4.-Secuestro',
       '5.-Delitos Contra La Libertad E Indemnidad Sexual',
       '5.1.-Agresión Sexual Con Penetración',
       '6.-Robos Con Violencia E Intimidación',
       '7.- Robos Con Fuerza En Domicilios, Establecimientos Y Otras Instalaciones',
       '7.1.-Robos Con Fuerza En Domicilios', '8.-Hurtos',
       '9.-Sustracciones De Vehículos', '10.-Tráfico De Drogas',
       '1. Homicidios Dolosos Y Asesinatos Consumados',
       '2. Homicidios Dolosos Y Asesinatos En Grado Tentativa',
       '3. Delitos Graves Y Menos Graves De Lesiones Y Riña Tumultuaria',
       '4. Secuestro', '5. Delitos Contra La Libertad Sexual',
       '6. Robos Con Violencia E Intimidación',
       '7. Robos Con Fuerza En Domicilios, Establecimientos Y Otras Instalaciones',
       '8. Hurtos', '9. Sustra

In [389]:
def generalizacion_tipologia_penal(df):
    mapa_delitos = {
    # Homicidios
    "1.-Homicidios Dolosos Y Asesinatos Consumados": "Homicidios",
    "1. Homicidios Dolosos Y Asesinatos Consumados": "Homicidios",
    "2.-Homicidios Dolosos Y Asesinatos En Grado Tentativa": "Homicidios",
    "2. Homicidios Dolosos Y Asesinatos En Grado Tentativa": "Homicidios",

    # Lesiones
    "3.-Delitos Graves Y Menos Graves De Lesiones Y Riña Tumultuaria": "Lesiones",
    "3. Delitos Graves Y Menos Graves De Lesiones Y Riña Tumultuaria": "Lesiones",

    # Secuestro
    "4.-Secuestro": "Secuestro",
    "4. Secuestro": "Secuestro",

    # Delitos sexuales
    "5.-Delitos Contra La Libertad E Indemnidad Sexual": "Delitos Sexuales",
    "5. Delitos Contra La Libertad Sexual": "Delitos Sexuales",
    "5.1.-Agresión Sexual Con Penetración": "Delitos Sexuales",
    "5.2.-Resto De Delitos Contra La Libertad E Indemnidad Sexual": "Delitos Sexuales",
    "5.2.-Resto De Delitos Contra La Libertad Sexual": "Delitos Sexuales",

    # Robos con violencia
    "6.-Robos Con Violencia E Intimidación": "Robos Con Violencia",
    "6. Robos Con Violencia E Intimidación": "Robos Con Violencia",

    # Robos con fuerza
    "7.- Robos Con Fuerza En Domicilios, Establecimientos Y Otras Instalaciones": "Robos Con Fuerza",
    "7. Robos Con Fuerza En Domicilios, Establecimientos Y Otras Instalaciones": "Robos Con Fuerza",
    "7.1.-Robos Con Fuerza En Domicilios": "Robos Con Fuerza",

    # Hurtos
    "8.-Hurtos": "Hurtos",
    "8. Hurtos": "Hurtos",

    # Sustracciones
    "9.-Sustracciones De Vehículos": "Sustraccion De Vehiculos",
    "9. Sustracciones De Vehículos": "Sustraccion De Vehiculos",

    # Tráfico de drogas
    "10.-Tráfico De Drogas": "Trafico De Drogas",
    "10. Tráfico De Drogas": "Trafico De Drogas",

    # Daños (categoría antigua)
    "7.-Daños": "Resto De Criminalidad Convencional"
    }

    df["Delitos"] = df["Tipología penal"].map(mapa_delitos)
    
    return df

df_generalizacion_delitos = generalizacion_tipologia_penal(df_tipologia_penal_normalizado)
df_generalizacion_delitos


,Geografía,Tipología penal,Periodos:,Total,Provincia,Delitos
0,Alcalá de Henares,1.-Homicidios Dolosos Y Asesinatos Consumados,Enero-junio 2020,0,Madrid,Homicidios
1,Alcalá de Henares,2.-Homicidios Dolosos Y Asesinatos En Grado Te...,Enero-junio 2020,1,Madrid,Homicidios
2,Alcalá de Henares,3.-Delitos Graves Y Menos Graves De Lesiones Y...,Enero-junio 2020,17,Madrid,Lesiones
3,Alcalá de Henares,4.-Secuestro,Enero-junio 2020,1,Madrid,Secuestro
4,Alcalá de Henares,5.-Delitos Contra La Libertad E Indemnidad Sexual,Enero-junio 2020,18,Madrid,Delitos Sexuales
...,...,...,...,...,...,...
143,San Sebastián de los Reyes,8.-Hurtos,Enero-marzo 2016,584,Madrid,Hurtos
150,Torrejón de Ardoz,7.-Daños,Enero-marzo 2016,217,Madrid,Resto De Criminalidad Convencional
151,Torrejón de Ardoz,8.-Hurtos,Enero-marzo 2016,416,Madrid,Hurtos
158,Valdemoro,7.-Daños,Enero-marzo 2016,119,Madrid,Resto De Criminalidad Convencional


In [390]:
sorted(df_generalizacion_delitos["Periodos:"].unique())

df_generalizacion_delitos["Periodos:"].str.startswith(" ").sum()

df_generalizacion_delitos["Periodos:"].str.strip().str.lower().value_counts()

df_generalizacion_delitos["Periodos:"].str.len().describe().T

df_generalizacion_delitos["Periodos:"].isna().sum()

df_generalizacion_delitos["Periodos:"].eq("").sum()


#Que valores de la columna "Periodos:" no cumplen el formato correcto:
#Que empiecen con una palabra
#Guion obligatorio
#Otra palabra
#Espacio
#Un año de 4 digitos
df_generalizacion_delitos[
    ~df_generalizacion_delitos["Periodos:"].str.match(r"^[A-Za-zÁÉÍÓÚáéíóúñÑ]+-[A-Za-zÁÉÍÓÚáéíóúñÑ]+ \d{4}$")
]["Periodos:"].unique()
print(df.columns.tolist())


['Geografía', 'Tipología penal', 'Periodos:', 'Total', 'Provincia']


In [391]:
def normalizacion_año_ultimo_trimestre(df):
    df = df.copy()
    #Normalizar texto del periodo
    periodos = (
    df["Periodos:"]
    .str.lower()
    .str.replace(r"\s*[-–]\s*", "-", regex=True)  # " - " o "–" → "-"
    .str.replace(r"\s+", " ", regex=True)
    .str.strip())
    
    #Extraccion del año
    df["Año"] = periodos.str.extract(r"(\d{4})$")
    
    #Extraer solo la parte del periodo (antes del año)
    df["Periodo"] = periodos.str.extract(r"^([a-záéíóúñ\- ]+)")
    df["Periodo"] = df["Periodo"].str.strip().str.title()

    # Filtrar solo el acumulado anual (Enero-Diciembre) 
    df = df[df["Periodo"] == "Enero-Diciembre"]
   
    #Ordenamos df
    df = df.sort_values(["Geografía", "Delitos", "Año","Provincia"])

    return df

df_periodos = normalizacion_año_ultimo_trimestre(df_generalizacion_delitos)

df_periodos





,Geografía,Tipología penal,Periodos:,Total,Provincia,Delitos,Año,Periodo
4,Alcalá de Henares,5.-Delitos Contra La Libertad E Indemnidad Sexual,Enero-diciembre 2017,53,Madrid,Delitos Sexuales,2017,Enero-Diciembre
5,Alcalá de Henares,5.1.-Agresión Sexual Con Penetración,Enero-diciembre 2017,2,Madrid,Delitos Sexuales,2017,Enero-Diciembre
4,Alcalá de Henares,5.-Delitos Contra La Libertad E Indemnidad Sexual,Enero-diciembre 2018,48,Madrid,Delitos Sexuales,2018,Enero-Diciembre
5,Alcalá de Henares,5.1.-Agresión Sexual Con Penetración,Enero-diciembre 2018,5,Madrid,Delitos Sexuales,2018,Enero-Diciembre
4,Alcalá de Henares,5.-Delitos Contra La Libertad E Indemnidad Sexual,Enero-diciembre 2019,51,Madrid,Delitos Sexuales,2019,Enero-Diciembre
...,...,...,...,...,...,...,...,...
677,Villaviciosa de Odón,9. Sustracciones De Vehículos,enero-diciembre 2024,8,Madrid,Sustraccion De Vehiculos,2024,Enero-Diciembre
522,Villaviciosa de Odón,10.-Tráfico De Drogas,Enero-diciembre 2021,3,Madrid,Trafico De Drogas,2021,Enero-Diciembre
659,Villaviciosa de Odón,10. Tráfico De Drogas,Enero-diciembre 2022,4,Madrid,Trafico De Drogas,2022,Enero-Diciembre
678,Villaviciosa de Odón,10. Tráfico De Drogas,enero-diciembre 2023,6,Madrid,Trafico De Drogas,2023,Enero-Diciembre


In [392]:
len(df_periodos["Delitos"].unique())

10

### Informacion
-Agrupamos y dentro de cada grupo sumamos la columna "Total". Por que?:
-Antes teniamos un mismo delito con diferentes nombres, como los categorizamos en delitos mas "generales" tenemos que sumarlos para obtener el valor real de esa categoria delictiva general

In [393]:
def pivot_delitos(df):
    df_suma = df.groupby(["Geografía", "Delitos", "Año", "Provincia"], as_index=False )["Total"].sum()

    df_pivot = df_suma.pivot_table(
        index=["Geografía", "Año", "Provincia"],
        columns="Delitos", 
        values="Total", 
        aggfunc="sum", 
        fill_value=0
    ).reset_index()

    columnas_delitos = df_pivot.columns.difference(["Geografía", "Año", "Provincia"])

    df_pivot["Total Delitos"] = df_pivot[columnas_delitos].sum(axis=1)
    
    df_pivot.columns.name = None

    return df_pivot

df_suma_tipologias = pivot_delitos(df_periodos)
df_suma_tipologias

,Geografía,Año,Provincia,Delitos Sexuales,Homicidios,Hurtos,Lesiones,Resto De Criminalidad Convencional,Robos Con Fuerza,Robos Con Violencia,Secuestro,Sustraccion De Vehiculos,Trafico De Drogas,Total Delitos
0,Alcalá de Henares,2016,Madrid,0,0,2801,0,1566,0,0,0,0,0,4367
1,Alcalá de Henares,2017,Madrid,55,5,2676,38,0,678,291,0,277,40,4060
2,Alcalá de Henares,2018,Madrid,53,4,2709,50,0,640,236,0,230,39,3961
3,Alcalá de Henares,2019,Madrid,55,4,2739,48,0,579,227,0,232,49,3933
4,Alcalá de Henares,2020,Madrid,52,2,1826,43,0,430,151,1,166,41,2712
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263,Villanueva de la Cañada,2024,Madrid,12,0,231,15,0,101,15,0,2,2,378
264,Villaviciosa de Odón,2021,Madrid,15,0,274,13,0,173,53,0,6,3,537
265,Villaviciosa de Odón,2022,Madrid,19,0,320,8,0,182,49,0,8,4,590
266,Villaviciosa de Odón,2023,Madrid,11,0,301,17,0,234,37,0,7,6,613


In [394]:
df_suma_tipologias.columns.tolist()


['Geografía',
 'Año',
 'Provincia',
 'Delitos Sexuales',
 'Homicidios',
 'Hurtos',
 'Lesiones',
 'Resto De Criminalidad Convencional',
 'Robos Con Fuerza',
 'Robos Con Violencia',
 'Secuestro',
 'Sustraccion De Vehiculos',
 'Trafico De Drogas',
 'Total Delitos']

In [395]:
def pipeline_provincia(df, mapa_normalizacion, lista_municipios):
    df = limpiar_geografia_general(df)
    df = limpiar_tipologia_penal(df)

    # Normalización de nombres de municipios (si hay mapa)
    if mapa_normalizacion:
        df = normalizacion_municipios(df, mapa_normalizacion)

    # Filtrado por lista oficial de municipios
    if lista_municipios:
        df = filtrar_municipios(df, lista_municipios)

    df = normalizar_texto_tipologia_penal(df)
    df = generalizacion_tipologia_penal(df)
    df = normalizacion_año_ultimo_trimestre(df)
    df = pivot_delitos(df)

    return df


df_valencia_limpio = pipeline_provincia(df_valencia,mapa_normalizacion_valencia, lista_municipios_valencia)

df_barcelona_limpio = pipeline_provincia(df_barcelona,{},[])

df_madrid_limpio = pipeline_provincia(df_madrid, {}, [])
df_madrid_limpio.columns


Index(['Geografía', 'Año', 'Provincia', 'Delitos Sexuales', 'Homicidios',
       'Hurtos', 'Lesiones', 'Resto De Criminalidad Convencional',
       'Robos Con Fuerza', 'Robos Con Violencia', 'Secuestro',
       'Sustraccion De Vehiculos', 'Trafico De Drogas', 'Total Delitos'],
      dtype='object')

In [396]:
df_final = pd.concat([df_valencia_limpio, df_barcelona_limpio, df_madrid_limpio], ignore_index=True)
df_final.to_csv("../data/processed/df_final.csv", index=False)


In [397]:
df_final


,Geografía,Año,Provincia,Delitos Sexuales,Homicidios,Hurtos,Lesiones,Resto De Criminalidad Convencional,Robos Con Fuerza,Robos Con Violencia,Secuestro,Sustraccion De Vehiculos,Trafico De Drogas,Total Delitos
0,Alaquàs,2021,Valencia,4,1,205,8,0,100,14,0,10,8,350
1,Alaquàs,2022,Valencia,5,0,240,14,0,99,21,0,16,7,402
2,Alaquàs,2023,Valencia,13,0,178,9,0,60,35,1,8,7,311
3,Alaquàs,2024,Valencia,9,2,188,9,0,77,22,0,12,12,331
4,Alboraia / Alboraya,2021,Valencia,3,0,391,4,0,86,32,0,16,3,535
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
771,Villanueva de la Cañada,2024,Madrid,12,0,231,15,0,101,15,0,2,2,378
772,Villaviciosa de Odón,2021,Madrid,15,0,274,13,0,173,53,0,6,3,537
773,Villaviciosa de Odón,2022,Madrid,19,0,320,8,0,182,49,0,8,4,590
774,Villaviciosa de Odón,2023,Madrid,11,0,301,17,0,234,37,0,7,6,613


In [56]:
df_final['Año'] = df_final['Año'].astype(int)